# Engagement 2: Mobile Money Agent Network Clean-Up

**Ramadhan Fatuma Soud**
1 Million Devs for Africa Software Engineering course - 1 Million Devs Cohort

A mobile money provider runs agents across five counties. The regional manager wants to know which agents and counties handle the most deposits and withdrawals. The legacy extract holds deliberate quality problems. The notebook audits them and cleans them with `clean_agent_data()` from `cleaning_pipeline.py`. It then merges the agent register and builds the county and weekly summaries. A Data Quality Report closes the notebook.

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

os.makedirs("outputs", exist_ok=True)

## 2. Load the raw files

In [2]:
raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")
raw_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

print("Transactions:", raw_txns.shape)
print("Agents:", raw_agents.shape)

Transactions: (612, 5)
Agents: (20, 3)


In [3]:
print(raw_txns.dtypes.to_string())

txn_id          str
agent_id        str
txn_type        str
amount      float64
txn_date        str


In [4]:
print(raw_txns.head(4).to_string(index=False))

 txn_id agent_id txn_type  amount   txn_date
TX00001    AG009  deposit   353.0 2026-03-03
TX00002    AG020  deposit  1238.0 2026-04-27
TX00003    AG008 DEPOSIT   9129.0 2026-04-29
TX00004    AG009  deposit   652.0 2026-04-24


The extract holds 612 rows and 5 columns. The register lists 20 agents. Pandas reads the text N/A as a missing value by default. A second read with `keep_default_na=False` (the `raw_text` table) keeps the original text visible so the audit can separate blank cells from N/A text.

## 3. Data-quality audit

In [5]:
dates = pd.to_datetime(raw_txns["txn_date"].str.replace("/", "-"), errors="coerce")
joined = raw_txns.merge(raw_agents, on="agent_id", how="left")
before_onboarding = int((dates.values < pd.to_datetime(joined["onboarded"]).values).sum())

amount_num = pd.to_numeric(raw_text["amount"], errors="coerce")
q1, q3 = amount_num.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)

rows = [
    ("Exact duplicate rows", "Uniqueness", "all columns", int(raw_txns.duplicated().sum())),
    ("Repeated txn_id after dedup", "Uniqueness", "txn_id", int(raw_txns.drop_duplicates()["txn_id"].duplicated().sum())),
    ("Mixed case or spacing", "Consistency", "txn_type", int((~raw_txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum())),
    ("Unknown agent_id", "Consistency", "agent_id", int((~raw_txns["agent_id"].isin(raw_agents["agent_id"])).sum())),
    ("Blank amount", "Completeness", "amount", int((raw_text["amount"] == "").sum())),
    ("Text N/A in amount", "Validity", "amount", int((raw_text["amount"] == "N/A").sum())),
    ("Slash date format", "Validity", "txn_date", int(raw_txns["txn_date"].str.contains("/").sum())),
    ("Zero or negative amount", "Validity", "amount", int((amount_num <= 0).sum())),
    ("Repeated extreme value", "Accuracy", "amount", int((amount_num == amount_num.max()).sum())),
    ("Amount above IQR fence", "Accuracy", "amount", int((amount_num > upper_fence).sum())),
    ("Dated before onboarding", "Timeliness", "txn_date", before_onboarding),
]
audit = pd.DataFrame(rows, columns=["issue", "dimension", "column", "rows_affected"])
print("Audit rules checked:", len(audit))
print("Rules with problems:", int((audit["rows_affected"] > 0).sum()))

Audit rules checked: 11
Rules with problems: 9


In [6]:
print(audit.iloc[0:4].to_string(index=False))

                      issue   dimension      column  rows_affected
       Exact duplicate rows  Uniqueness all columns             11
Repeated txn_id after dedup  Uniqueness      txn_id              1
      Mixed case or spacing Consistency    txn_type            384
           Unknown agent_id Consistency    agent_id              8


In [7]:
print(audit.iloc[4:8].to_string(index=False))

                  issue    dimension   column  rows_affected
           Blank amount Completeness   amount             15
     Text N/A in amount     Validity   amount              7
      Slash date format     Validity txn_date              5
Zero or negative amount     Validity   amount              0


In [8]:
print(audit.iloc[8:11].to_string(index=False))

                  issue  dimension   column  rows_affected
 Repeated extreme value   Accuracy   amount              5
 Amount above IQR fence   Accuracy   amount             40
Dated before onboarding Timeliness txn_date              0


In [9]:
print(audit.groupby("dimension")["rows_affected"].sum().to_string())

dimension
Accuracy         45
Completeness     15
Consistency     392
Timeliness        0
Uniqueness       12
Validity         12


The audit checks 11 rules across the six dimensions and finds problems in 9 of them. The two clean checks (no zero or negative amounts and no transaction dated before its agent joined) are listed with zero rows so the governance team can see they were tested. The counts describe the raw extract. Duplicate copies of a row inflate some of them.

## 4. Run the cleaning pipeline

In [10]:
clean_df, log = clean_agent_data(raw_txns, raw_agents)
print("Clean table:", clean_df.shape)

Clean table: (600, 10)


In [11]:
for line in log[0:3]:
    print("-", line)

- Start: 612 transaction rows and 20 agents
- Removed 11 exact duplicate rows
- Removed 1 repeated txn_id rows (kept the copy with a known agent)


In [12]:
for line in log[3:6]:
    print("-", line)

- Standardised 376 txn_type values to Deposit or Withdrawal (0 unmapped)
- Amount: 20 missing (includes N/A read as missing) and 0 invalid text values set to NaN
- Dates: converted 5 slash dates to ISO format (0 unreadable)


In [13]:
for line in log[6:9]:
    print("-", line)

- Outliers: fences -1,928 to 6,084, flagged 40 rows for review
- Imputed 20 amounts with the median 1,776.50 and set amount_imputed to True
- Merge: 600 rows before and 600 after a left join; 7 rows have an unknown agent (county set to Unknown)


In [14]:
for line in log[9:]:
    print("-", line)

- Finish: 600 clean rows


The function applies every rule in code. No row number or manual edit appears anywhere in it. The log lines above go straight into the governance review.

## 5. Before and after row counts

In [15]:
print("Rows before cleaning:", len(raw_txns))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(raw_txns) - len(clean_df))
print("Share kept:", np.round(len(clean_df) / len(raw_txns) * 100, 1), "%")

Rows before cleaning: 612
Rows after cleaning: 600
Rows removed: 12
Share kept: 98.0 %


The pipeline removes 12 rows from 612 and keeps 600. All 12 are duplicates: 11 exact copies and 1 repeated `txn_id` whose copy carried a wrong agent. Every valid transaction stays in the table.

In [16]:
print("Duplicate txn_id:", clean_df["txn_id"].duplicated().sum())
print("txn_type values:", sorted(clean_df["txn_type"].unique()))
print("amount dtype:", clean_df["amount"].dtype, "| missing:", clean_df["amount"].isna().sum())
print("txn_date dtype:", clean_df["txn_date"].dtype)
print("Date range:", clean_df["txn_date"].min().date(), "to", clean_df["txn_date"].max().date())

Duplicate txn_id: 0
txn_type values: ['Deposit', 'Withdrawal']
amount dtype: float64 | missing: 0
txn_date dtype: datetime64[us]
Date range: 2026-03-01 to 2026-04-29


In [17]:
print(clean_df[["amount_imputed", "is_outlier", "agent_known"]].sum().to_string())

amount_imputed     20
is_outlier         40
agent_known       593


## 6. Completeness: missing and invalid amounts

In [18]:
imp = clean_df[clean_df["amount_imputed"]]
total = clean_df["amount"].sum()
print("Imputed rows:", len(imp), "| share of rows:", np.round(len(imp) / len(clean_df) * 100, 1), "%")
print("Median used:", imp["amount"].iloc[0])
print("Imputed value:", imp["amount"].sum(), "| share of value:", np.round(imp["amount"].sum() / total * 100, 1), "%")

Imputed rows: 20 | share of rows: 3.3 %
Median used: 1776.5
Imputed value: 35530.0 | share of value: 1.3 %


**Decision.** Twenty amounts are missing or invalid after duplicates go. That is 3.3% of rows. The pipeline flags each one in `amount_imputed` and fills it with the median of the observed amounts.

**Why the median.** The transaction happened. Only its amount is lost. Dropping the rows would erase 20 real transactions from every count. The mean sits far above the median because of the extreme values, so the median is the safer centre. The imputed value adds KES 35,530 and makes up 1.3% of total value. Anyone who needs verified values only can filter on `amount_imputed == False`.

## 7. Outliers: the IQR method and the KES 250,000 values

In [19]:
obs = clean_df.loc[~clean_df["amount_imputed"], "amount"]
q1, q3 = obs.quantile([0.25, 0.75])
iqr = q3 - q1
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Lower fence:", q1 - 1.5 * iqr, "| Upper fence:", q3 + 1.5 * iqr)
print("Flagged:", clean_df["is_outlier"].sum(), "of", len(clean_df))

Q1: 1077.0 | Q3: 3080.0 | IQR: 2003.0
Lower fence: -1927.5 | Upper fence: 6084.5
Flagged: 40 of 600


In [20]:
spike_value = clean_df["amount"].max()
spikes = clean_df[clean_df["amount"] == spike_value]
print(spikes[["txn_id", "agent_id", "county", "amount"]].to_string(index=False))

 txn_id agent_id  county   amount
TX00307    AG012  Kisumu 250000.0
TX00316    AG014 Nairobi 250000.0
TX00425    AG005 Mombasa 250000.0
TX00464    AG009  Nakuru 250000.0
TX00595    AG016  Kiambu 250000.0


In [21]:
others = clean_df[clean_df["amount"] < spike_value]
print("Spike rows:", len(spikes), "| distinct agents:", spikes["agent_id"].nunique(), "| distinct counties:", spikes["county"].nunique())
print("Largest other amount:", others["amount"].max())
print("Spike is", np.round(spike_value / others["amount"].max(), 1), "times the next largest")
print("Spike share of total value:", np.round(spikes["amount"].sum() / total * 100, 1), "%")
print("Other flagged rows:", int(clean_df["is_outlier"].sum() - len(spikes)))

Spike rows: 5 | distinct agents: 5 | distinct counties: 5
Largest other amount: 28847.0
Spike is 8.7 times the next largest
Spike share of total value: 46.5 %
Other flagged rows: 35


The IQR method flags 40 rows above the upper fence of KES 6,084. Five of them share one exact value: KES 250,000. They sit with five different agents in five different counties. The next largest amount is KES 28,847 so each spike is 8.7 times bigger. The five rows carry 46.5% of all transaction value. A round figure that repeats this often points to a system test value or a keying error. A genuine customer pattern rarely repeats a round figure across agents.

The other 35 flagged rows are natural high values. Mobile money amounts have a long right tail and the largest of these is KES 28,847. They stay in the data.

**Recommendation.** Keep the five 250,000 rows in the file with `is_outlier` set to True. Hold them in quarantine and ask the operations team to match each one against the core ledger and the agent float records. Report totals with and without them until the check finishes. Delete or correct a row only after the data owner signs off. Restore it to normal reporting if the ledger confirms it.

In [22]:
without = clean_df[clean_df["amount"] < spike_value]
compare = pd.DataFrame({
    "total_all": clean_df.groupby("county")["amount"].sum(),
    "total_without_spikes": without.groupby("county")["amount"].sum(),
}).round(0).astype(int)
print(compare.to_string())

         total_all  total_without_spikes
county                                  
Kiambu      411788                161788
Kisumu      512795                262795
Mombasa     517578                267578
Nairobi     705626                455626
Nakuru      513484                263484
Unknown      27156                 27156


## 8. Merge validation: unknown agents

In [23]:
unknown = clean_df[~clean_df["agent_known"]]
print("Clean rows after merge:", len(clean_df), "| unique txn_id:", clean_df["txn_id"].nunique())
print("Rows an inner join would keep:", int(clean_df["agent_known"].sum()))
print("Transactions with unknown agents:", len(unknown))
print("Unknown agent ids:", sorted(unknown["agent_id"].unique()))
print("County label used:", unknown["county"].unique().tolist())
print("Unknown value:", unknown["amount"].sum(), "|", np.round(unknown["amount"].sum() / total * 100, 1), "% of total value")

Clean rows after merge: 600 | unique txn_id: 600
Rows an inner join would keep: 593
Transactions with unknown agents: 7
Unknown agent ids: ['AG099']
County label used: ['Unknown']
Unknown value: 27156.0 | 1.0 % of total value


The merge is a left join with the many_to_one validation check. Many transactions point to one agent and the register lists each agent once. The row count stays at 600 before and after the join.

Seven transactions reference agent AG099 and the register has no such agent. The pipeline keeps them with county set to `Unknown` and `agent_known` set to False. Money moved in those transactions. An inner join would drop 7 rows and KES 27,156. Guessing a county would invent data. The list goes to the agent onboarding team to confirm whether AG099 is an unregistered agent or a keying error. One AG099 row was a copy of TX00405. The original row carried agent AG014 and the pipeline kept that one.

## 9. Summaries

In [24]:
county_summary = (clean_df.groupby(["txn_type", "county"])
    .agg(total_value=("amount", "sum"), transactions=("txn_id", "count"), avg_value=("amount", "mean"))
    .round(0).astype(int).reset_index())
print("Deposits")
print(county_summary[county_summary["txn_type"] == "Deposit"].drop(columns="txn_type").to_string(index=False))

Deposits
 county  total_value  transactions  avg_value
 Kiambu       334737            43       7785
 Kisumu       436420            74       5898
Mombasa       148986            72       2069
Nairobi       543590           118       4607
 Nakuru       184890            60       3082
Unknown        25279             6       4213


In [25]:
print("Withdrawals")
print(county_summary[county_summary["txn_type"] == "Withdrawal"].drop(columns="txn_type").to_string(index=False))

Withdrawals
 county  total_value  transactions  avg_value
 Kiambu        77050            27       2854
 Kisumu        76374            37       2064
Mombasa       368592            61       6042
Nairobi       162036            62       2613
 Nakuru       328594            39       8426
Unknown         1877             1       1877


In [26]:
verified = clean_df[clean_df["amount"] < spike_value].pivot_table(
    index="county", columns="txn_type", values="amount", aggfunc="sum").round(0).astype(int)
print("Total value without the five spike rows")
print(verified.to_string())

Total value without the five spike rows
txn_type  Deposit  Withdrawal
county                       
Kiambu      84737       77050
Kisumu     186420       76374
Mombasa    148986      118592
Nairobi    293590      162036
Nakuru     184890       78594
Unknown     25279        1877


**Deposits.** Nairobi leads with KES 543,590 across 118 transactions. It still leads at KES 293,590 after the spike row goes. Kisumu follows at KES 186,420 without its spike.

**Withdrawals.** Mombasa (KES 368,592) and Nakuru (KES 328,594) top the raw table. Each holds one 250,000 spike. Without the spikes Nairobi leads withdrawals at KES 162,036. Mombasa drops to KES 118,592 and Nakuru to KES 78,594.

**Reading the table.** Nairobi handles the most transactions overall (180 of 600). Its lead holds in both views. The Mombasa and Nakuru withdrawal ranks depend on one unverified amount each. Report them only after the ledger check.

In [27]:
clean_df["week_start"] = clean_df["txn_date"].dt.to_period("W").dt.start_time.dt.strftime("%Y-%m-%d")
weekly = clean_df.pivot_table(index="week_start", columns="county", values="amount",
                              aggfunc="sum", fill_value=0)
print("Weekly pivot shape:", weekly.shape)
print("Pivot total:", weekly.to_numpy().sum(), "| clean total:", clean_df["amount"].sum())

Weekly pivot shape: (10, 6)
Pivot total: 2688427.0 | clean total: 2688427.0


In [28]:
print("KES thousands")
print((weekly / 1000).round(0).astype(int).iloc[0:3].to_string())

KES thousands
county      Kiambu  Kisumu  Mombasa  Nairobi  Nakuru  Unknown
week_start                                                   
2026-02-23       0       2        7        7       3        0
2026-03-02      18     288      286       69      57        0
2026-03-09     268      23       30       53      29        0


In [29]:
print((weekly / 1000).round(0).astype(int).iloc[3:6].to_string())

county      Kiambu  Kisumu  Mombasa  Nairobi  Nakuru  Unknown
week_start                                                   
2026-03-16      22      28       25       43      40        2
2026-03-23      12      27       17       41      19        0
2026-03-30      23      41       38      283      18        9


In [30]:
print((weekly / 1000).round(0).astype(int).iloc[6:9].to_string())

county      Kiambu  Kisumu  Mombasa  Nairobi  Nakuru  Unknown
week_start                                                   
2026-04-06      15      15       47       57     277       10
2026-04-13      12      29       27       56      26        0
2026-04-20      25      37       17       65      31        2


In [31]:
print((weekly / 1000).round(0).astype(int).iloc[9:10].to_string())

county      Kiambu  Kisumu  Mombasa  Nairobi  Nakuru  Unknown
week_start                                                   
2026-04-27      18      21       24       31      15        5


Weeks run Monday to Sunday and `week_start` shows the Monday. The extract covers 1 March to 29 April 2026. The first week holds only Sunday 1 March and the last week holds three days. The pivot total matches the clean table total. The big weekly jumps line up with the spike rows: Kisumu and Mombasa in the week of 2 March, Kiambu in the week of 9 March, Nairobi in the week of 30 March and Nakuru in the week of 6 April. Outside those weeks the counties move in a narrow band.

## 10. Save the clean file

In [32]:
clean_df.drop(columns="week_start").to_csv("outputs/agent_transactions_clean.csv", index=False)
check = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Saved rows:", len(check), "| columns:", len(check.columns))
print(list(check.columns))
print(check["txn_date"].head(3).tolist())

Saved rows: 600 | columns: 10
['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'amount_imputed', 'county', 'onboarded', 'agent_known']
['2026-03-03', '2026-04-27', '2026-04-29']


## 11. Data Quality Report

The provider asked for a trusted view of agent deposits and withdrawals. The legacy extract held 612 rows from 20 registered agents across five counties. The audit tested 11 rules and found problems in 9. The pipeline returns 600 clean rows.

**Completeness.** Twenty amounts were missing after duplicates went. Fifteen blank cells and seven N/A texts made up the 22 raw cases. Each one carries `amount_imputed` set to True and the median of KES 1,776.50. They hold 1.3% of total value.

**Uniqueness.** The extract held 11 exact duplicate rows and 1 repeated `txn_id` (TX00405, listed under AG014 and AG099). All 12 were removed. Row count fell from 612 to 600. The clean file has no repeated `txn_id`.

**Validity.** Seven amounts held the text N/A. Pandas reads it as missing so the pipeline checks `isna()` and invalid strings together. Five dates used slashes and now follow the ISO format. No amount is zero or negative and every date converted.

**Consistency.** Mixed case and trailing spaces affected 384 `txn_type` values. Every row now reads Deposit or Withdrawal. Eight raw rows pointed to agent AG099 and seven remain after deduplication. They keep county `Unknown` and `agent_known` set to False.

**Accuracy.** Five rows hold exactly KES 250,000 in five counties. Each is 8.7 times the next largest amount and together they carry 46.5% of value. They stay in the file with `is_outlier` set to True and sit in quarantine until the ledger check. Another 35 flagged rows are natural high values and stay in normal reporting.

**Timeliness.** Every transaction falls between 1 March and 29 April 2026. Every transaction follows its agent onboarding date.

**Privacy.** Customer phone numbers would be personal data under the Kenya Data Protection Act 2019 because they identify a living person. Transaction date and amount become personal data too once a phone number links them. Agent IDs and counties describe individual agents and also count when an agent runs as a sole trader. I would remove phone numbers from the analysis extract because county and agent summaries need none. Where a number must stay I would replace it with a keyed hash and store the key apart from the data. Access would go to named staff with encryption at rest. The raw extract stays off GitHub through `.gitignore`. The provider needs a documented lawful basis, a retention limit and a data protection impact assessment. The provider also registers with the Office of the Data Protection Commissioner and sends data outside Kenya only with safeguards in place.